# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Priyansh-rath18/flyrank-internship-/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking / scoring.**

The triage-ranking lane's real question is "which declining page should an editor fix first?" — a which-ones-first question, which maps to ranking/scoring, not classification or clustering. There's no natural fixed set of classes to sort pages into, and a plain yes/no label would throw away the ordering an editor actually needs when hours are limited. Scoring each page and ranking the queue lets an editor work top-down, and lets us measure success directly with precision@K against FlyRank's existing rule-based baseline.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Predict:** a per-page decline-severity / priority score used to rank the triage queue.

**Label source:** it comes from an observed outcome, not a defined rule. The label is built from the trailing-window traffic change already in the data — clicks/sessions in `*_last_30d` vs `*_prev_30d` (what `trend_pct` measures) — i.e. a real, measured drop in traffic, not FlyRank's hand-written health-score or quick-win tag. Using the rule's own output as the label would just teach the model to reproduce the rule; using the observed traffic change lets the model potentially find patterns the rule misses.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50.** Of the top 50 pages the model ranks highest-priority, what fraction are pages that actually show a real, observed decline in the holdout window? This is defensible because it mirrors how editors actually work (a fixed queue of hours per week, not every page), and it's directly comparable to FlyRank's existing hand-rule baseline, which the docs report at roughly 0.24 precision@50. "Good" means beating that baseline by a clear margin (directionally, in the 0.6–0.75 range reported for the trained models on this data) — computed honestly on a client-holdout split, not on training data.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one (content page, snapshot).** Each row of `content_refresh_anonymized.csv` is a single client page (`content_id`, nested under `client_id`) with its traffic/engagement metrics over a fixed trailing window. For this lane the slice is the subset of rows already flagged as declining (`trend_direction == "down"`) — those are the candidates a triage-ranking model would actually be scoring and ordering for an editor.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

url = "https://raw.githubusercontent.com/Priyansh-rath18/flyrank-internship-/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

lane_df = df[df['trend_direction'] == 'down'].copy()

print(f"full dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"declining-lane slice (one row = one content page snapshot): {lane_df.shape[0]} rows")
lane_df[['content_id', 'client_id', 'trend_direction', 'trend_pct', 'clicks_last_30d', 'clicks_prev_30d']].head()

full dataset: 30000 rows, 44 columns
declining-lane slice (one row = one content page snapshot): 16262 rows


,content_id,client_id,trend_direction,trend_pct,clicks_last_30d,clicks_prev_30d
0,content_304f48230142,client_f369cb89fc,down,-41.4,2,13
1,content_a1fb4e703a9e,client_4e07408562,down,-57.7,2,1
2,content_9aa793d4d895,client_7f2253d7e2,down,-60.9,1,3
4,content_d99b7a2d90ca,client_3fdba35f04,down,-34.7,10,2
5,content_d4084a4bc775,client_f369cb89fc,down,-38.9,0,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Too many interacting, shifting signals for one if-statement.** A fixed rule (like FlyRank's current health-score/quick-win tag) has to pick a small, static set of thresholds — e.g. "position worse than X and clicks down Y%" — but decline actually shows up through many correlated signals at once (word count, freshness/`days_since_last_update`, `ai_traffic_pct`, `engagement_rate`, `competition_level`, seasonality) whose relative importance differs by client and drifts over time. Real pages break the rule's assumptions in different combinations (e.g. a page can look fine on position but be losing engaged sessions, or vice versa), so a hand-written rule either misses real decliners or over-flags noisy ones. That's exactly the gap the docs' own numbers show: the hand-rule baseline hits precision@50 ≈ 0.24 while a trained model reaches ≈ 0.68–0.74 on the same data — the messiness is real, and a model that can weigh many signals together captures it where a single if-statement can't.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.